# 04 — Strategy Backtest & Performance Attribution

Applies the regime classification to a live trading strategy and evaluates performance vs the JSE Top 40 buy-and-hold benchmark.

**Transaction costs:** 30bps round trip + 5bps slippage (realistic JSE assumptions)
**Benchmark:** JSE Top 40 Total Return (buy-and-hold)
**Risk-free rate:** 7.5% (SARB repo rate proxy)

**Prerequisites:** Run notebooks 01–03 first.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from src.data.loader import get_price_data, load_config
from src.strategy.router import RegimeRouter
from src.strategy.backtest import RegimeBacktest
from src.utils.plotting import RegimePlotter
from src.utils.metrics import performance_summary, regime_performance_breakdown

%matplotlib inline
config = load_config('../config/settings.yaml')
config['paths']['figures'] = '../results/figures/'
config['paths']['processed_data'] = '../data/processed/'
print("Ready")

In [ ]:
prices = get_price_data(config)
processed = Path('../data/processed/')

# Load regime outputs — prefer Bayesian, fall back to classical
try:
    regime_probs = pd.read_csv(processed / 'regime_probs_bayesian.csv',
                                index_col='date', parse_dates=True)
    confidence = pd.read_csv(processed / 'position_confidence.csv',
                              index_col='date', parse_dates=True).squeeze()
    print("Using Bayesian regime probabilities.")
except FileNotFoundError:
    print("Bayesian results not found — using classical. Run notebook 03 for full results.")
    regime_probs = pd.read_csv(processed / 'regime_probs_classical.csv',
                                index_col='date', parse_dates=True)
    confidence = pd.Series(1.0, index=regime_probs.index, name='confidence')

print(f"Regime probs shape: {regime_probs.shape}")
print(f"Date range: {regime_probs.index[0].date()} → {regime_probs.index[-1].date()}")

## Position Sizing

In [ ]:
router = RegimeRouter(config)
positions_df = router.compute_positions(regime_probs, confidence, prices['index'])

print("Position sizing summary:")
print(f"  Mean position:          {positions_df['position'].mean():.2%}")
print(f"  Fully invested (>90%):  {(positions_df['position'] > 0.9).mean():.1%}")
print(f"  Cash (<10%):            {(positions_df['position'] < 0.1).mean():.1%}")
print()
print("Regime transition matrix:")
print(router.regime_transition_summary(regime_probs).round(3))

## Run Backtest

In [ ]:
backtest = RegimeBacktest(config)
strategy_returns = backtest.run(
    prices['index'],
    regime_probs,
    confidence,
    label='JSE Bayesian Regime Strategy'
)
print(f"Backtest complete: {len(strategy_returns)} trading days")

In [ ]:
backtest.performance_report()

## Visualisations

In [ ]:
plotter = RegimePlotter(config)
res = backtest.results['JSE Bayesian Regime Strategy']

_ = plotter.backtest_results(
    strategy_returns=res['strategy_returns'],
    benchmark_returns=res['benchmark_returns'],
    positions=positions_df['position'],
    regime_probs=regime_probs,
    confidence=confidence,
    save=True,
)

In [ ]:
_ = plotter.regime_return_distributions(
    returns=res['benchmark_returns'],
    regime_probs=regime_probs,
    save=True,
)

In [ ]:
_ = plotter.rolling_performance(
    strategy_returns=res['strategy_returns'],
    benchmark_returns=res['benchmark_returns'],
    window=252,
    save=True,
)

## Known SA Market Event Analysis

In [ ]:
# How did the strategy perform during known SA stress events?
events = {
    'GFC':         ('2008-06-01', '2009-06-01'),
    'Nenegate':    ('2015-09-01', '2016-03-01'),
    'COVID':       ('2020-01-01', '2020-09-01'),
    'Fed Tighten': ('2022-01-01', '2022-12-31'),
}

sr = res['strategy_returns']
br = res['benchmark_returns']

print(f"{'Event':<15} {'Strategy':>12} {'Benchmark':>12} {'Alpha':>10}")
print("-" * 52)
for event, (start, end) in events.items():
    s = sr[start:end]
    b = br[start:end]
    if len(s) == 0:
        continue
    strat_cum = (1 + s).prod() - 1
    bench_cum = (1 + b).prod() - 1
    alpha = strat_cum - bench_cum
    print(f"{event:<15} {strat_cum:>12.2%} {bench_cum:>12.2%} {alpha:>10.2%}")